# Step 4: The improved architecture
**YOLOv8m + residual CBAM attention + learnable weighted multi-scale fusion.**

| Change | Where | Idea |
|---|---|---|
| Residual CBAM (Woo et al., ECCV 2018) | after SPPF and after each neck output P3/P4/P5 (4 blocks) | channel attention (which channels), then spatial attention (where) |
| Weighted concat | every neck `Concat` (4 of them) | each incoming scale gets a learnable non-negative weight, normalised, then concatenated |

Residual gate: `y = x + gamma * (CBAM(x) - x)` with `gamma = 0` at the start, so the pretrained YOLOv8m features pass through unchanged when fine-tuning begins.
Differences from the CBAM paper: SiLU in the shared MLP (YOLO convention) and the residual gate. Reduction ratio 16 and the 7x7 spatial kernel follow the paper.

In [ ]:
# Set True for a fast CPU sanity run (3 epochs, 320 px, 25% of train). False = full experiment.
QUICK_TEST = False
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # works from VS Code either way
sys.path.insert(0, str(ROOT / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config(quick=QUICK_TEST)
RUNS = md.ROOT / "runs"
YOLO_DIR = md.ROOT / cfg["dataset"]["yolo_dir"]
DATA_YAML = YOLO_DIR / "data.yaml"
import torch, yaml
names = list(yaml.safe_load(open(DATA_YAML))["names"].values()) if DATA_YAML.exists() else ["c0", "c1", "c2", "c3"]
nc = len(names)

## 4.1 The generated model YAML

In [ ]:
path, old2new = models.save_cfg("proposed", attention=True, nc=nc)
print(open(path).read())

## 4.2 Parameters and compute: baseline vs improved

In [ ]:
import pandas as pd
from mri_detection import modules as cm
def stats(net, imgsz=640):
    from ultralytics.utils.torch_utils import get_flops
    return {"Params (M)": sum(p.numel() for p in net.parameters()) / 1e6, "GFLOPs": get_flops(net, imgsz)}
base = models.build_network("baseline", nc=nc)
prop = models.build_network("improved", nc=nc)
tab = pd.DataFrame({"Baseline YOLOv8m": stats(base), "Improved YOLOv8m": stats(prop)})
tab["Change"] = tab["Improved YOLOv8m"] - tab["Baseline YOLOv8m"]
tab.round(3)

## 4.3 Sanity checks

In [ ]:
prop.eval()
with torch.no_grad():
    out = prop(torch.zeros(1, 3, 640, 640))
print("Detect-head feature maps (P3, P4, P5):", [tuple(o.shape) for o in out[1]])
print("CBAM blocks:", sum(isinstance(m, cm.CBAMAttention) for m in prop.modules()),
      "| WeightedConcat layers:", sum(isinstance(m, cm.WeightedConcat) for m in prop.modules()))

# 1) At initialisation the CBAM block must be the identity (gamma = 0)
x = torch.randn(2, 192, 20, 20)
blk = cm.CBAMAttention(192).eval()
assert torch.allclose(blk(x), x), "CBAM should be identity at init"
print("CBAM is the identity at init: OK")

# 2) At initialisation WeightedConcat must equal plain concatenation
a, b = torch.randn(1, 8, 4, 4), torch.randn(1, 8, 4, 4)
wc = cm.WeightedConcat(1, 2)
assert torch.allclose(wc([a, b]), torch.cat([a, b], 1), atol=1e-3), "WeightedConcat should ~ Concat at init"
print("WeightedConcat ~ plain Concat at init: OK")

# 3) The gate and the fusion weights are trainable
x = torch.randn(2, 192, 20, 20); blk.train(); blk(x).sum().backward()
print("gamma receives gradient:", blk.gamma.grad is not None and blk.gamma.grad.abs().item() > 0)

## 4.4 Pretrained-weight transfer
The improved model has new layers, so layer numbers shift. During training a callback maps each COCO-pretrained YOLOv8m tensor to its new index and copies it if the shape matches. Here the mapping is checked without training:

In [ ]:
from ultralytics import YOLO
old = YOLO(models.pretrained_weights()).model.float().state_dict()
new_sd = prop.state_dict()
remapped = {}
for k, v in old.items():
    p = k.split(".")
    if p[0] == "model" and p[1].isdigit(): p[1] = str(old2new.get(int(p[1]), int(p[1])))
    remapped[".".join(p)] = v
ok = {k for k, v in remapped.items() if k in new_sd and v.shape == new_sd[k].shape}
print(f"{len(ok)} of {len(old)} pretrained tensors fit the improved model; the {len(old) - len(ok)} that do not are the 80-class COCO head")
print("Tensors only in the improved model (start from initial values):", len(new_sd) - len(ok))

Next: `05_train_improved.ipynb`.